# 15.4 能只啟動少數 experts 嗎？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**k=1會讓所有廚師都做嗎？

top-k只送給分數最高的少數廚師。選中哪些是離散決定，gate權重仍是連續數值；兩件事的梯度性質不同。

**把直覺寫成數學：**topk取每token最大的k個p值與expert索引。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
prob = torch.tensor([[0.1, 0.7, 0.2], [0.6, 0.3, 0.1]])
weights, chosen = prob.topk(2, dim=-1)
print(chosen, weights)

**如何讀結果：**這裡還沒dispatch；下一節計算gate合併。選中indices本身不可微。

**只改一件事：**只把k改為1。
